# Practical 2 — Build your own Marmara earthquake catalog
By Oliver Fabisch and Sebastian Nunez Jara

Turn local recordings into **AI picks → PyOcto and GaMMA events → local magnitudes**.

We study **24 September 2019, 07:30-11:45 UTC** using eleven TU and five KO stations. This includes the foreshock at **08:00:22 UTC**. The AFAD catalog reports it as **Mw 4.6** (https://deprem.afad.gov.tr/event-detail/71140).

You will compare two pick thresholds, inspect the resulting earthquake candidates, and calculate a local-magnitude estimate from the same recordings.

Source: [the 2019 Silivri sequence](https://doi.org/10.1093/gji/ggaa469).

**Reading**

* Which Picker Fits my Data? Munchmeyer et al., 2022. https://doi.org/10.1029/2021JB023499
* Earthquake Phase Association Using a Bayesian Gaussian Mixture Model. Zhu et al., 2022. https://doi.org/10.1029/2021JB023249
* PyOcto: A high-throughput seismic phase associator. Munchmeyer, 2024. https://doi.org/10.26443/seismica.v3i1.1130
* Benchmarking seismic phase associators: Insights from synthetic scenarios. Puente-Huerta et al., 2025. https://doi.org/10.26443/seismica.v4i2.1559
* PhaseNet: a deep-neural-network-based seismic arrival-time picking method. Zhu and Beroza, 2019. https://doi.org/10.1093/gji/ggy423



## Prepare the workspace

This notebook runs in the course environment `applied-seismology` (see `README.md` in the course folder); the first cell below checks it.

The waveforms, station metadata, AFAD reference catalog and pretrained PhaseNet weights are downloaded automatically from the [GitHub Release](https://github.com/sebmagia/applied_seismology_rwth/releases/tag/marmara-catalog-practical-v1) of this practical, into the folder `marmara_catalog_data/` next to this notebook (or in advance with `python download_data.py catalog`). The exercise uses CPU inference and does not train a network. All timestamps below are UTC.

Sources: [ObsPy](https://docs.obspy.org/), [SeisBench pretrained weights](https://seisbench.readthedocs.io/en/stable/pages/models/pretrained_models.html).

In [ ]:
# Check that this notebook runs in the course environment (see README.md).
import sys
from importlib.util import find_spec

missing = [pkg for pkg in ('obspy', 'pyproj', 'torch', 'seisbench', 'pyocto', 'gamma', 'ipympl') if find_spec(pkg) is None]
if missing:
    raise ModuleNotFoundError(
        f"Not installed for this kernel ({sys.executable}): {', '.join(missing)}. "
        "Start Jupyter from the course environment, or select its kernel (see README.md).")
print("Course environment OK:", sys.prefix)

In [ ]:
# Record the versions used in this run.
import sys
import numpy, scipy, pandas, matplotlib, obspy, seisbench, pyproj
print("python", sys.version.split()[0])
for m in (numpy, scipy, pandas, matplotlib, obspy, seisbench, pyproj):
    print(m.__name__, m.__version__)

### Import the packages

Imports make the installed packages available to the cells below (they do not install anything).

In [ ]:
# Interactive figures (zoom, pan) with ipympl, for all the figures of this notebook.
%matplotlib widget

from pathlib import Path
import time
import urllib.request
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.max_open_warning"] = 200   # interactive figures stay open: closing one would remove it from the notebook
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment
from scipy.signal import spectrogram
from pyproj import Transformer

from obspy import UTCDateTime, Stream, Inventory, read, read_inventory
from obspy.geodetics import gps2dist_azimuth
import seisbench.models as sbm
from tqdm.auto import tqdm
import pyocto
from gamma.utils import association, estimate_eps

### Download the data

The next cell downloads the waveforms, station metadata, AFAD reference catalog and PhaseNet weights the first time, into `marmara_catalog_data/` next to this notebook; afterwards it finds them there. The cell after it fixes the study interval.

In [ ]:
RELEASE_BASE = (
    "https://github.com/sebmagia/applied_seismology_rwth/"
    "releases/download/marmara-catalog-practical-v1"
)

DATA_ROOT = Path.cwd() / "marmara_catalog_data"   # next to this notebook
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# Download and extract each archive only if its folder is not there yet.
for name in ["data", "models"]:
    if (DATA_ROOT / name).exists():
        print(f"{name}/ already available")
        continue

    zip_path = DATA_ROOT / f"{name}.zip"
    print(f"Downloading {RELEASE_BASE}/{name}.zip ...")
    urllib.request.urlretrieve(f"{RELEASE_BASE}/{name}.zip", zip_path)

    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_ROOT)
    zip_path.unlink(missing_ok=True)

print("Dataset root:", DATA_ROOT)
for d in sorted(DATA_ROOT.iterdir()):
    print(" ", d.name)

In [ ]:
# Fixed study interval. Extra samples support arrivals and filtering near its edges.
t0 = UTCDateTime("2019-09-24T07:30")
t1 = UTCDateTime("2019-09-24T11:45")
read_start, read_end = t0-60, t1+60  # Processing padding in seconds.
print("Study interval:", t0, "to", t1)

## Part 1 — The data: stations, reference events and recordings

### 1.1 Read waveforms and station metadata

MiniSEED contains recorded samples in instrument counts. StationXML supplies coordinates and the instrument responses needed to recover physical motion. Our channels are HHZ, HHN and HHE: vertical, north and east.

The files contain hourly segments. We merge them into one continuous trace per channel so that SeisBench can process them in one pass. Short gaps between segments are filled with the last recorded value (`fill_value="latest"`); data missing at the start or end of a recording stay missing.

Sources: [SeisBench processing](https://seisbench.readthedocs.io/en/stable/_modules/seisbench/models/base.html), [ObsPy waveform formats](https://docs.obspy.org/packages/obspy.io.mseed.html), [ObsPy merge](https://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.merge.html).

In [ ]:
# Read metadata first, then the waveform samples; neither operation downloads data.
inventory = Inventory(networks=[], source="Prepared classroom dataset")
for path in sorted((DATA_ROOT / "data/stations").glob("*.xml")):
    inventory += read_inventory(str(path))
print(inventory)
raw = Stream()
for path in sorted((DATA_ROOT / "data/waveforms").glob("*/*.mseed")):
    # ObsPy selects the requested interval while reading each local file.
    raw += read(str(path), starttime=read_start, endtime=read_end)
raw.trim(read_start, read_end)
print(len(raw), "waveform segments loaded into RAM")

# Merge the hourly segments into one trace per channel so that SeisBench can handle them.
raw.merge(method=1, fill_value="latest", interpolation_samples=0)
print(raw.__str__(extended=True))

In [ ]:
# Prepared receiver coordinates; elevation is the sensor height in meters above sea level.
# The IDs identify network, station and location, just as in the SeisBench picks.
stations = pd.read_csv(DATA_ROOT/"data/stations.csv")
display(stations.head(n=20))

In [ ]:
# Select AFAD by origin interval and region, without a magnitude/type filter.
afad = pd.read_csv(DATA_ROOT / "data/reference/afad.csv")
in_interval = afad.time.between(t0.timestamp, t1.timestamp, inclusive="left")
in_region = afad.latitude.between(40.6, 41.05) & afad.longitude.between(27.7, 28.9) ## Sea of Marmara
afad = afad[in_interval & in_region].reset_index(drop=True)
# The largest AFAD event in the selection is the Mw 4.6 foreshock: our reference event.
foreshock = afad.loc[afad.magnitude.idxmax()]
print(len(stations), "stations and", len(afad), "AFAD events")
display(afad[["EventID", "origin_time_utc", "magnitude", "type"]].sort_values("origin_time_utc"))

In [ ]:
def map_limits(tables, padding=0.025):
    """Fit a map around every plotted event, with a small margin in degrees."""
    locations = pd.concat(tables, ignore_index=True)
    longitude_limits = (locations.longitude.min()-padding, locations.longitude.max()+padding)
    latitude_limits = (locations.latitude.min()-padding, locations.latitude.max()+padding)
    return longitude_limits, latitude_limits

# Correct the apparent map shape for the shorter east-west degree at this latitude.
map_aspect = 1 / np.cos(np.deg2rad(40.8))
afad_xlim, afad_ylim = map_limits([afad])

### 1.2 Station geometry and reference epicenters

**Left:** triangles show the full station network; circles show AFAD epicenters. This view helps us judge whether stations surround the source region or mainly lie on one side.

**Right:** the same AFAD earthquakes are enlarged without distant stations setting the map limits. The star marks the large foreshock. Circle color shows AFAD magnitude; the catalog includes both ML and Mw, so color is only a rough size guide here.

In [ ]:
COAST_FILE = DATA_ROOT / "data/marmara_coastline.csv"
coastlines = pd.read_csv(COAST_FILE)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax in axes:
    ax.set_facecolor("lightblue")
    for _, seg in coastlines.groupby("segment"):
        ax.plot(seg["longitude"], seg["latitude"], color="black", linewidth=0.8, zorder=2)

# Regional view: keep the full network visible.
for network, group in stations.groupby(stations.id.str.split(".").str[0]):
    axes[0].scatter(group.longitude, group.latitude, marker="^", label=network)
    for _, sta in group.iterrows():
        axes[0].annotate(sta.id, xy=(sta.longitude, sta.latitude), xytext=(4, 4),
                         textcoords="offset points", fontsize=7, ha="left", va="bottom")
axes[0].scatter(afad.longitude, afad.latitude, color="0.4", s=25, label="AFAD")
axes[0].text(foreshock.longitude + 0.06, foreshock.latitude - 0.12,
             "AFAD Mw 4.6 foreshock", fontsize=9, zorder=6)
axes[0].set_title("Station geometry: regional view")
axes[0].legend(fontsize=8)

# Local view: AFAD events determine the extent, so their cluster is readable.
points = axes[1].scatter(afad.longitude, afad.latitude, c=afad.magnitude,
                         s=45, cmap="viridis", edgecolors="black", linewidths=0.3)
axes[1].set(xlim=afad_xlim, ylim=afad_ylim, title="Zoom: AFAD reference events")
fig.colorbar(points, ax=axes[1], label="AFAD magnitude")

for ax in axes:
    ax.scatter(foreshock.longitude, foreshock.latitude, marker="*", s=180,
               color="red", edgecolors="black", zorder=5)
    ax.set(xlabel="Longitude (°)", ylabel="Latitude (°)")
    ax.set_aspect(map_aspect)
plt.show()

#### Exercise 1

* What are your thoughts on the station coverage for this sequence?
* What could be the main limitations of this station coverage, and how would you improve the monitoring?

### 1.3 Read the recording coverage

Each row is one station. A blue segment means samples exist; a blank interval means no data were recorded there (short gaps were already filled when merging). The red line marks the AFAD foreshock origin. The waterfall below shows the vertical recording at TU.ELBA over the complete interval.

In [ ]:
# Reference event: the AFAD foreshock.
print(foreshock)
ref_lat = float(foreshock["latitude"])
ref_lon = float(foreshock["longitude"])


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Great-circle distance in km using haversine formula.
    Inputs may be scalars or numpy arrays.
    """
    R = 6371.0  # Earth radius [km]

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2)**2
    )

    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
# Draw each continuous segment separately so that missing data stay visible.
# Stations are sorted by distance to the foreshock; this will help us later.
stations_plot = stations.copy()
stations_plot["distance_km"] = haversine_km(
    ref_lat, ref_lon,
    stations_plot["latitude"].to_numpy(),
    stations_plot["longitude"].to_numpy(),
)
stations_plot = (
    stations_plot
    .sort_values("distance_km", ascending=False)   # nearest station ends up on top
    .reset_index(drop=True)
)
print(stations_plot)

fig, ax = plt.subplots(figsize=(12, 8))
for offset, row in stations_plot.iterrows():
    for tr in raw.select(id=f"{row['id']}.HHZ"):
        left = (tr.stats.starttime - t0) / 60
        right = (tr.stats.endtime - t0) / 60
        ax.plot([left, right], [offset, offset], color="tab:blue", lw=3)

event_time = UTCDateTime(foreshock["time"])
ax.axvline((event_time - t0) / 60, color="red", ls="--", label="Reference event time")

ax.set(
    yticks=range(len(stations_plot)),
    yticklabels=[f"{row.id}  ({row.distance_km:.1f} km)" for _, row in stations_plot.iterrows()],
    xlabel="Minutes after selection start",
    ylabel="Station",
    title="Vertical-channel coverage, sorted by distance to reference event",
)
ax.tick_params(axis="y", labelsize=7)
plt.show()

### 1.4 See small arrivals in a waveform waterfall

Each row shows 15 minutes from the vertical component at TU.ELBA, filtered to 1–15 Hz. Read from left to right, then continue on the next row. The amplitude scale is shared across rows and uses ObsPy's percentile scaling: strong arrivals can overlap adjacent rows so that small signals remain visible. The star identifies the AFAD foreshock.

[ObsPy dayplot documentation](https://docs.obspy.org/packages/autogen/obspy.imaging.waveform.WaveformPlotting.html).

#### Exercise 2

How many potential events can you identify by eye? What happens if you change to stations that are farther away from the study region?

In [ ]:
# Filter a plotting copy; the raw stream remains unchanged for picking and magnitudes.
station = "ELBA"
waterfall = raw.select(station=station, channel="HHZ").copy()
waterfall.detrend("demean")
waterfall.filter("bandpass", freqmin=1, freqmax=15, corners=2, zerophase=True)

# handle=True keeps the figure: without it ObsPy closes the figure after drawing it, and an
# interactive (%matplotlib widget) figure then disappears from the notebook.
waterfall.plot(
    type="dayplot", interval=15,  # Minutes per row of the waterfall.
    starttime=t0, endtime=t1,
    timezone="UTC", time_offset=0,
    size=(1100, 750), right_vertical_labels=False,
    vertical_scaling_range=None,  # Percentile scaling makes weak arrivals visible; large ones may overlap.
    events=[{"time": UTCDateTime(float(foreshock.time)), "text": "AFAD Mw 4.6"}],
    title=f"{station}..HHZ — 24 September 2019, 1–15 Hz", handle=True);

## Part 2 — Learn what a pretrained picker has learned

A **dataset** contains examples; an **architecture** specifies the neural network; a set of **weights** records what training learned. Applying pretrained weights is inference. SeisBench supplies a common interface to datasets (`data`), networks (`models`) and training transformations (`generate`). We only need `models` here.

| Dataset / weights | Character of the examples | Relevance to transfer |
|---|---|---|
| INSTANCE | Italian earthquake and noise recordings, with arrival labels | Nearby-region crustal recordings provide a starting point. |
| STEAD | A broad collection of earthquake and noise waveforms | Instrument and noise diversity influence transfer. |
| ETHZ | Swiss regional seismic recordings | Depths and propagation paths can differ from Marmara. |
| GEOFON | Regional and teleseismic examples | Local S-arrival representation affects picking. |
| PhaseNet `original` | Original northern California training | Regional differences can change onset scores. |

We use **PhaseNet with INSTANCE weights**, trained on Italian earthquake and noise recordings. AFAD provides the reference events for evaluating the resulting catalog.

PhaseNet predicts a noise/P/S score for each sample. EQTransformer is another architecture that jointly addresses detection and picking; changing weights within PhaseNet is a different experiment from changing architecture. The pretrained weights are included in the exercise folder.

Sources: [SeisBench model tutorial](https://github.com/seisbench/seisbench/blob/main/examples/01b_model_api.ipynb), [dataset documentation](https://seisbench.readthedocs.io/en/stable/pages/documentation/data/waveform_datasets.html), [INSTANCE](https://essd.copernicus.org/articles/13/5509/2021/), [STEAD](https://github.com/smousavi05/STEAD), [PhaseNet](https://doi.org/10.1093/gji/ggy423), [EQTransformer](https://doi.org/10.1038/s41467-020-17591-w).

In [ ]:
model = sbm.PhaseNet.load( DATA_ROOT / "models/instance", version_str="2")  # Fixed weight version.
print("Architecture:", model.name)
print("Expected sampling rate:", model.sampling_rate, "Hz")
print("Input samples per window:", model.in_samples)

### 2.1 Predict P and S scores

SeisBench resamples and normalizes a copy of the input, then applies PhaseNet to overlapping windows. `strict=True` requires all three components. The model returns a score for each sample; **note:** a score is not a calibrated probability that an earthquake exists.

We can call `annotate` directly; no separate PyTorch context is needed. [SeisBench model tutorial](https://github.com/seisbench/seisbench/blob/main/examples/01b_model_api.ipynb).

In [ ]:
# `await model.annotate_async(...)` is SeisBench's `model.annotate(...)` run in Jupyter's own event loop.
# Inside Jupyter, `model.annotate(...)` starts a second loop, which can drop the cells queued by Run All.
start_clock = time.perf_counter()
annotations = await model.annotate_async(
    raw,  # SeisBench processes a copy and retains the original instrument counts.
    batch_size=64,  # Number of input windows processed together.
    overlap=1500,  # Shared samples between 3001-sample windows at 100 Hz.
    blinding=(250, 250),  # Ignore 2.5 s at each prediction-window edge.
    strict=True)  # Predict only where all three components are available.
print(f"Prediction took {time.perf_counter()-start_clock:.1f} seconds.")
# in that arrival, there could be noise, P wave arrival, S wave arrival
# P(noise) + P(arrival_p) * P(arrival_s) = 1.0
#P(p_arrival > 10%) with P(p_arrival > 30%)

### 2.2 Compare pick thresholds of 0.1 and 0.3

A low threshold retains weaker arrivals as well as more false triggers. Multi-station phase association then checks whether arrivals can describe a common source. A threshold of **0.1 for both P and S** is used in published PhaseNet–INSTANCE catalog workflows, including [Münchmeyer et al. (2025), Atacama](https://doi.org/10.1029/2025JB031256) and [the South Peru catalog study](https://doi.org/10.1093/gji/ggag119).

We compare 0.1 and 0.3 using the same scores and PyOcto settings. We continue with **0.1** for GaMMA and magnitude calculations.

In [ ]:
def make_pick_table(score_stream, threshold):
    """Convert each P/S score peak into one row with UTC and pandas timestamps."""
    rows = []

    for phase in ["P", "S"]:
        phase_scores = score_stream.select(channel=f"PhaseNet_{phase}")
        phase_picks = model.picks_from_annotations(
            phase_scores,
            threshold,
            phase
        )

        for pick in phase_picks:
            rows.append({
                "station": pick.trace_id,
                "time": pick.peak_time.timestamp,
                "phase": pick.phase,
                "probability": pick.peak_value
            })

    table = pd.DataFrame(
        rows,
        columns=["station", "time", "phase", "probability"]
    )

    # Add timezone-aware pandas datetime
    table["datetime"] = pd.to_datetime(
        table["time"],
        unit="s",
        utc=True
    )

    return table.sort_values("time").reset_index(drop=True)

In [ ]:
# Two explicit pick tables from exactly the same INSTANCE score curves.
picks = make_pick_table(annotations, 0.1)  # Main picks for the remaining workflow.
picks_03 = make_pick_table(annotations, 0.3)  # More restrictive comparison.
threshold = 0.1  # Used to draw the threshold on the score plot.
display(picks.head(n=25))
print("Picks at 0.1:", len(picks), "— picks at 0.3:", len(picks_03))

The helper converts each P/S score excursion into one peak arrival. Its trigger ends below half the chosen threshold. Keep the table row index: both associators will refer back to it.

In [ ]:
# Choose one station and a short interval around the AFAD foreshock.
station_id = "TU.ELBA."  # Example station for displaying the foreshock.
station_name = station_id.split(".")[1]
station_picks = picks[picks.station == station_id]
left = UTCDateTime(float(foreshock.time))-5
right = left+70  # A focused foreshock view, used for inspection only.

In [ ]:
# Overlay arrival picks on the raw recordings and their underlying model scores.
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True, constrained_layout=True)
for ax, component in zip(axes[:3], ["Z", "N", "E"]):
    for tr in raw.select(id=f"{station_id}.HH{component}").slice(left, right):
        ax.plot(tr.times(reftime=left), tr.data, color="0.25", lw=0.6)
    ax.set_ylabel(f"{component} counts")
for phase, color in [("P", "tab:blue"), ("S", "tab:orange")]:
    for score in annotations.select(station=station_name, channel=f"PhaseNet_{phase}").slice(left, right):
        axes[3].plot(score.times(reftime=left), score.data, color=color, label=phase)
    visible = station_picks[(station_picks.phase == phase) &
                            station_picks.time.between(left.timestamp, right.timestamp)]
    for pick_time in visible.time:
        for ax in axes[:3]:
            ax.axvline(pick_time-left.timestamp, color=color, lw=1, alpha=0.8)
axes[3].axhline(threshold, color="black", ls="--", label="Threshold")
axes[3].set(ylabel="Model score", ylim=(0, 1.05), xlabel=f"Seconds after {left}")
axes[3].legend(ncol=3)
axes[0].set_title(f"{station_id}: Waveforms and phase picks")
plt.show()

In [ ]:
# Reuse predictions: changing the threshold does not require rerunning the neural network.
comparison = []
for value in [0.10, 0.30]:
    trial = make_pick_table(annotations, value)
    comparison.append({"threshold": value, "P": sum(trial.phase == "P"),
                       "S": sum(trial.phase == "S")})
comparison = pd.DataFrame(comparison).set_index("threshold")
comparison.plot.bar(rot=0, ylabel="Number of picks", title="Same predictions, different thresholds")
plt.show()
display(comparison)

### 2.3 Same architecture, different training

The two panels show INSTANCE and original weights applied to the same foreshock snippet at one station. Keeping the architecture and waveform fixed highlights how training data influence the P and S score curves.

In [ ]:
# Keep the architecture fixed and change only the training weights.
original_model = sbm.PhaseNet.load( DATA_ROOT / "models/original", version_str="2")  # Fixed weight version.
# You can try also other different models and assess their performance! https://github.com/seisbench/seisbench/blob/main/examples/01b_model_api.ipynb
# Note that the geofon weights produce few S picks.
another_model_name = 'ethz'
# list_pretrained() and from_pretrained() need internet; downloaded weights are cached in ~/.seisbench
available_weights = sbm.PhaseNet.list_pretrained()
print(available_weights)
another_model = sbm.PhaseNet.from_pretrained(another_model_name)

print(model.weights_docstring)
print(original_model.weights_docstring)
print(another_model.weights_docstring)
snippet = raw.select(network="KO", station="SLVT").slice(left-30, right+30)
original_scores = await original_model.annotate_async(snippet, strict=True)   # see the prediction cell above
another_scores = await another_model.annotate_async(snippet, strict=True)

fig, axes = plt.subplots(3, 1, figsize=(11, 7.5), sharex=True, constrained_layout=True)
for ax, scores, label in zip(axes, [annotations, original_scores, another_scores], ["INSTANCE", "Original", f"{another_model_name}"]):
    for phase in ["P", "S"]:
        for tr in scores.select(station="SLVT", channel=f"PhaseNet_{phase}").slice(left, right):
            ax.plot(tr.times(reftime=left), tr.data, label=phase)
    ax.set(ylabel="Score", ylim=(0, 1.05), title=label)
    ax.legend()
axes[-1].set_xlabel("Seconds after foreshock display start")
plt.show()

## Part 3 — Turn picks into earthquakes: PyOcto and GaMMA

A pick is an arrival at one station. An earthquake is a source that explains arrivals at several stations. We apply **two associators to exactly the same PhaseNet picks**:

- **PyOcto** searches the source region and origin times for consistent groups of arrivals.
- **GaMMA** fits a Bayesian Gaussian mixture model, grouping picks using predicted travel times.

Both methods use the same constant P and S velocities and station coordinates. They can produce different catalogs even from identical picks. Here GaMMA uses arrival times only (`use_amplitude=False`); we do not ask GaMMA to estimate magnitudes.

We inspect both catalogs, then carry **only the PyOcto catalog** into the magnitude section and the reference catalog is reserved for comparison after association.

Sources: [PyOcto tutorial](https://github.com/seisbench/seisbench/blob/main/examples/03d_catalog_seisbench_pyocto.ipynb), [GaMMA tutorial](https://github.com/seisbench/seisbench/blob/main/examples/03c_catalog_seisbench_gamma.ipynb), [GaMMA implementation and parameters](https://github.com/AI4EPS/GaMMA), [Zhu et al. (2022)](https://doi.org/10.1029/2021JB023249).

### 3.1 Predict arrivals with a constant-velocity model

We use **Vp = 6.0 km/s** and **Vs = 3.5 km/s**. Travel time is source-station distance divided by the relevant wave speed. The same constant velocities are used throughout the source region.

The search covers 40.6-41.05° N, 27.7-28.9° E and depths of 0-40 km. Stations may lie outside the source box. `location_split_depth=12` and `location_split_return=10` control branch exploration; `min_node_size_location=0.2` sets the final cell threshold in km. The 0.2 km threshold specifies the size of the final search cells.

Depths close to the 0 or 40 km search limits can indicate weak depth constraints or a mismatch between the constant-velocity model and the travel paths. [PyOcto parameter guide](https://pyocto.readthedocs.io/en/stable/pages/parameters.html).

In [ ]:
velocity_model = pyocto.VelocityModel0D(
    p_velocity=6.0,  # P-wave speed, km/s.
    s_velocity=3.5,  # S-wave speed, km/s.
    tolerance=1.5,  # Travel-time uncertainty for the association search, seconds.
    association_cutoff_distance=150.0)  # Distance cutoff for initial grouping, km.

In [ ]:
associator = pyocto.OctoAssociator.from_area(
    lat=(40.6, 41.05), lon=(27.7, 28.9),  # Source region in degrees.
    zlim=(0, 40),  # Source depth bounds in km, positive downward.
    velocity_model=velocity_model,
    min_node_size=5.0,  # Association-cell threshold, km.
    min_node_size_location=0.2,  # Final location-cell threshold, km.
    location_split_depth=12,  # Subdivision depth when exploring location branches.
    location_split_return=10,  # Dense branch sampling; must be smaller than 12.
    time_before=90,  # Overlap of temporal search slices, seconds.
    n_picks=8,  # Minimum total arrivals per event.
    n_p_picks=3, n_s_picks=3,  # Minimum arrivals of each phase.
    n_p_and_s_picks=3,  # Minimum stations with both P and S.
    pick_match_tolerance=1.5,  # Accepted arrival mismatch, seconds.
    n_threads=2)  # CPU workers for PyOcto.

In [ ]:
# PyOcto uses local kilometer coordinates for its travel-time calculation.
octo_stations = stations[["id", "latitude", "longitude", "elevation"]].copy()
associator.transform_stations(octo_stations)  # Degrees/meters → local x/y/z in km.
display(octo_stations.head())

In [ ]:
# Each assignment links one input pick to one output event.
events, assignments = associator.associate(picks, octo_stations)
associator.transform_events(events)  # Add latitude, longitude and depth.
# Count assignments explicitly, independently of PyOcto column naming.
events["n_picks"] = events.idx.map(assignments.groupby("event_idx").size())

# Retain the target interval; padding was only needed for processing.
events = events[events.time.between(t0.timestamp, t1.timestamp, inclusive="left")].copy()
assignments = assignments[assignments.event_idx.isin(events.idx)].copy()
events = events.sort_values("time")
print(f"{len(events)} candidate earthquakes in our selected interval.")
display(events[["idx", "time", "latitude", "longitude", "depth", "n_picks"]].head(n=39))
print(events["n_picks"].median())

### 3.2 Event catalogs at two picking thresholds

Run PyOcto once more with the 0.3 picks and identical settings. Both maps share their extent and depth colors. Open circles show AFAD events of every magnitude type.

In [ ]:
# Change only the input pick threshold; retain the same associator.
events_03, assignments_03 = associator.associate(picks_03, octo_stations)
associator.transform_events(events_03)
events_03 = events_03[events_03.time.between(
    t0.timestamp, t1.timestamp, inclusive="left")].copy()
assignments_03 = assignments_03[assignments_03.event_idx.isin(events_03.idx)].copy()
threshold_catalogs = {"INSTANCE 0.1": events, "INSTANCE 0.3": events_03}
print("Events at 0.1:", len(events), "— events at 0.3:", len(events_03))

In [ ]:
# A common map extent avoids making either catalog appear more tightly clustered.
xlim, ylim = map_limits([afad, events, events_03])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
for ax, (label, table) in zip(axes, threshold_catalogs.items()):
    ax.scatter(afad.longitude, afad.latitude, facecolors="none", edgecolors="black",
               s=60, label="AFAD", zorder=3)
    points = ax.scatter(table.longitude, table.latitude, c=table.depth,
                        cmap="viridis_r", vmin=0, vmax=40, s=25)
    ax.set(xlim=xlim, ylim=ylim, title=f"{label}: {len(table)} candidates",
           xlabel="Longitude (°)", ylabel="Latitude (°)")
    ax.set_aspect(map_aspect)
    ax.legend()
fig.colorbar(points, ax=axes, label="Estimated depth (km)")
plt.show()

### 3.3 Check timing residuals

A residual is **observed arrival minus predicted arrival**, in seconds. Zero means that the observed and predicted arrivals agree. Positive values mean that arrivals are later than predicted and negative values mean that they are earlier. A shift of most P or S residuals to one side can indicate a velocity-model bias.

In [ ]:
# Compare the timing fit of P and S arrivals assigned by PyOcto.
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
for phase in ["P", "S"]:
    values = assignments.loc[assignments.phase == phase, "residual"]
    ax.hist(values, bins=np.linspace(-1.5, 1.5, 31), alpha=0.6, label=phase)
ax.set(xlabel="Observed − predicted arrival (s)", ylabel="Number of picks")
ax.legend()
plt.show()
print(f"Median absolute residual: {assignments.residual.abs().median():.2f} s")

### 3.4 Give the same picks to GaMMA

The two packages expect different column names. We translate the table, keeping its row index unchanged so every output assignment can be traced back to the original pick. GaMMA expects lower-case `p` / `s` and timestamps as datetime objects. We reuse PyOcto's local coordinate system: **all x, y and z values are in km**, and station z is negative elevation.

In [ ]:
# Rename columns to the names GaMMA expects; the row order stays unchanged.
gamma_picks = pd.DataFrame({
    "id": picks.station,
    "timestamp": pd.to_datetime(picks.time, unit="s", utc=True).dt.tz_localize(None),
    "type": picks.phase.str.lower(), "prob": picks.probability})
gamma_stations = octo_stations[["id", "x", "y", "z"]].rename(
    columns={"x": "x(km)", "y": "y(km)", "z": "z(km)"})
display(gamma_picks.head())
# Set the grouping radius from station spacing and P speed, in time-scaled units.
gamma_radius = estimate_eps(gamma_stations, vp=6.0)
print(f"DBSCAN grouping radius: {gamma_radius:.1f}")

### 3.5 GaMMA parameters

DBSCAN groups arrivals using their times and station coordinates divided by Vp. The radius therefore combines temporal and spatial proximity; it is not an arrival-residual tolerance. `estimate_eps` estimates a suitable scale from the station spacing. A radius that is too small separates arrivals from the same earthquake into groups with too few picks.

BGMM then fits possible earthquakes within each group. `oversample_factor` supplies extra initial candidates; the mixture can reduce their number. `covariance_prior` regularizes arrival scatter, while `max_sigma11` limits absolute arrival residuals in this pinned version. The amplitude entry is unused because association uses only arrival times. We additionally require three stations with both P and S in the accepted output. [GaMMA implementation](https://github.com/AI4EPS/GaMMA/blob/80394dd4a4c29450da599f507916ecf1db6203dd/gamma/utils.py).

In [ ]:
gamma_config = {
    "dims": ["x(km)", "y(km)", "z(km)"],  # Three spatial coordinates.
    "x(km)": tuple(associator.xlim), "y(km)": tuple(associator.ylim),  # Same area.
    "z(km)": (0, 40),  # Same source-depth search bounds as PyOcto.
    "vel": {"p": 6.0, "s": 3.5},  # Constant speeds in km/s, as for PyOcto.
    "use_amplitude": False,  # Association only; no GaMMA magnitude estimate.
    "use_dbscan": True,  # Pre-cluster to reduce computation.
    "dbscan_eps": gamma_radius,  # Radius for time and distance/Vp grouping.
    "dbscan_min_samples": 3,  # Minimum neighborhood support for DBSCAN.
    "oversample_factor": 4,  # Start with extra candidate mixture components.
    "covariance_prior": [1.0, 1.0],  # Arrival variance prior (s^2); unused amplitude prior.
    "min_picks_per_eq": 8,  # Minimum total arrivals.
    "min_p_picks_per_eq": 3, "min_s_picks_per_eq": 3,  # Support from both phases.
    "max_sigma11": 1.5,  # Absolute arrival-residual cutoff (s), in this pinned version.
    "ncpu": 1}  # One worker is straightforward in a notebook.
# Optimizer bounds for x, y, depth, and origin respectively; origin is unbounded.
gamma_config["bfgs_bounds"] = (
    gamma_config["x(km)"], gamma_config["y(km)"], (0, 40), (None, None))

In [ ]:
# Run the mixture fit, then keep its events and pick assignments in separate tables.
np.random.seed(42)  # Make the initial mixture-model choices reproducible.
start_clock = time.perf_counter()
gamma_sources, gamma_links = association(
    gamma_picks, gamma_stations, gamma_config, method="BGMM")
gamma_runtime = time.perf_counter()-start_clock
print(f"GaMMA finished in {gamma_runtime:.1f} seconds.")
gamma_events = pd.DataFrame(gamma_sources)
gamma_assignments = pd.DataFrame(
    gamma_links, columns=["pick_idx", "event_idx", "prob_gamma"])

### 3.6 Put GaMMA's output into readable tables

A GaMMA assignment score and a PhaseNet pick score describe different decisions. We keep them in separate columns. Event IDs are also local to each method: event 5 from GaMMA is **not necessarily** event 5 from PyOcto.

In [ ]:
gamma_events = gamma_events.rename(columns={"event_index": "idx", "z(km)": "depth"})
print(gamma_events)
# Amplitudes are disabled: remove GaMMA's unused magnitude placeholder (999).
#gamma_events = gamma_events.drop(columns=["magnitude", "sigma_amp", "cov_time_amp"])
# Convert model origin timestamps into the same UTC seconds used by PyOcto.
gamma_events["time"] = [UTCDateTime(value).timestamp for value in gamma_events.time]

# Change local kilometer coordinates back into longitude and latitude for maps.
to_geographic = Transformer.from_crs(associator.crs, "EPSG:4326", always_xy=True)
gamma_events["longitude"], gamma_events["latitude"] = to_geographic.transform(
    gamma_events["x(km)"].to_numpy(), gamma_events["y(km)"].to_numpy())
gamma_assignments = gamma_assignments.merge(
    picks, left_on="pick_idx", right_index=True, validate="many_to_one")
# Count stations at which both a P and an S pick support each event.
phase_counts = gamma_assignments.groupby(["event_idx", "station"]).phase.nunique()
paired_stations = phase_counts.eq(2).groupby("event_idx").sum()
gamma_events["paired_stations"] = gamma_events.idx.map(paired_stations).fillna(0)

In [ ]:
# Keep the same geographical box, origin interval and paired-station minimum.
keep = (gamma_events.time.between(t0.timestamp, t1.timestamp, inclusive="left") &
        gamma_events.latitude.between(40.6, 41.05) &
        gamma_events.longitude.between(27.7, 28.9) &
        gamma_events.paired_stations.ge(3))
gamma_events = gamma_events[keep].sort_values("time").copy()
gamma_assignments = gamma_assignments[gamma_assignments.event_idx.isin(gamma_events.idx)].copy()
gamma_events["n_picks"] = gamma_events.idx.map(gamma_assignments.groupby("event_idx").size())
print(f"PyOcto: {len(events)} candidates; GaMMA: {len(gamma_events)} candidates.")
display(gamma_events[["idx", "time", "latitude", "longitude", "depth", "n_picks"]].head())

### 3.7 Compare pick support: PyOcto and GaMMA

The histogram counts events with a given number of assigned P and S picks. Both methods use the same INSTANCE picks at threshold 0.1. A shift to the right means more arrivals per candidate. P and S at one station contribute two picks. Each curve summarizes the pick support across its catalog.

In [ ]:
# Use identical bins so that the two distributions are directly comparable.
maximum_picks = max(events.n_picks.max(), gamma_events.n_picks.max())
pick_bins = np.arange(7.5, maximum_picks+4.5, 4)
fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
for name, catalog in [("PyOcto", events), ("GaMMA", gamma_events)]:
    ax.hist(catalog.n_picks, bins=pick_bins, histtype="step", linewidth=2,
            label=f"{name}: {len(catalog)} candidates")
ax.set(xlabel="Assigned P + S picks per candidate", ylabel="Number of candidates")
ax.legend()
plt.show()

### 3.8 Compare both associators with AFAD on the same zoom

Both maps use **identical limits**, fitted around the combined AFAD and "automatic" catalogs. Open black circles are AFAD epicenters; colored dots are the "automatic" candidates. The matching step below uses origin time and distance to link individual events.

The third panel counts events cumulatively: each step adds one event. A higher "automatic" curve means more candidates, which can include real small events, false detections or splits of one earthquake into several candidates.

In [ ]:
# Use all three catalogs to set a shared zoom; no candidate is cropped away.
comparison_xlim, comparison_ylim = map_limits([afad, events, gamma_events])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for ax, table, label in zip(axes[:2], [events, gamma_events], ["PyOcto", "GaMMA"]):
    ax.scatter(afad.longitude, afad.latitude, facecolors="none", edgecolors="black",
               s=70, linewidths=0.9, label="AFAD", zorder=3)
    ax.scatter(table.longitude, table.latitude, s=25, alpha=0.8, label=label)
    ax.set(xlim=comparison_xlim, ylim=comparison_ylim, xlabel="Longitude (°)",
           ylabel="Latitude (°)", title=f"{label} and AFAD: same zoom")
    ax.set_aspect(map_aspect)
    ax.legend(fontsize=8)
# Sorting origins makes the cumulative curve increase by one at each event.
for table, label in [(events, "PyOcto"), (gamma_events, "GaMMA"), (afad, "AFAD")]:
    ordered = table.sort_values("time")
    axes[2].step((ordered.time-t0.timestamp)/60, np.arange(1, len(ordered)+1),
                 where="post", label=label)
axes[2].set(xlim=(0, (t1-t0)/60), xlabel="Minutes after selection start",
            ylabel="Cumulative event count", title="Cumulative catalog growth")
axes[2].legend()
plt.show()

### 3.9 Compare events by shared picks

We propose a match when a PyOcto event and a GaMMA event share at least four **identical input picks** and have origin times within three seconds. For a simple one-to-one comparison we accept the largest shared-pick counts first.

In [ ]:
# Join assignments by pick ID to find arrivals used by both methods.
shared = assignments[["pick_idx", "event_idx"]].merge(
    gamma_assignments[["pick_idx", "event_idx"]], on="pick_idx", suffixes=("_octo", "_gamma"))
overlap = shared.groupby(["event_idx_octo", "event_idx_gamma"]).size().reset_index(name="shared_picks")
octo_times = events.set_index("idx").time
gamma_times = gamma_events.set_index("idx").time
overlap["origin_difference_s"] = (overlap.event_idx_octo.map(octo_times) -
                                   overlap.event_idx_gamma.map(gamma_times))
candidates = overlap[(overlap.shared_picks >= 4) & (overlap.origin_difference_s.abs() <= 3)]

# A candidate can enter only one pair; prefer pairs sharing the most arrivals.
used_octo, used_gamma, matched_rows = set(), set(), []
for pair in candidates.sort_values("shared_picks", ascending=False).itertuples(index=False):
    if pair.event_idx_octo in used_octo or pair.event_idx_gamma in used_gamma:
        continue
    matched_rows.append(pair._asdict())
    used_octo.add(pair.event_idx_octo)
    used_gamma.add(pair.event_idx_gamma)
method_matches = pd.DataFrame(matched_rows, columns=candidates.columns)

### 3.10 Agreement between the two automatic methods

The bars count PyOcto–GaMMA matches under the shared-pick rule. The histogram shows PyOcto origin minus GaMMA origin for those pairs. A value of +1 s means PyOcto placed the earthquake one second later. One PyOcto candidate can correspond to two GaMMA candidates. Our one-to-one rule leaves at least one unmatched in that case, even if both contain relevant arrivals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
labels = ["Matched pairs", "PyOcto unmatched", "GaMMA unmatched"]
axes[0].bar(labels, [len(method_matches), len(events)-len(method_matches),
                      len(gamma_events)-len(method_matches)])
axes[0].tick_params(axis="x", labelsize=8)
axes[0].set_ylabel("Count")
axes[1].hist(method_matches.origin_difference_s, bins=np.linspace(-3, 3, 16))
axes[1].set(xlabel="PyOcto − GaMMA origin (s)", ylabel="Matched pairs")
plt.show()
print("Unmatched means no match under our rule; it does not mean false.")

### 3.11 Foreshock recovery

The AFAD foreshock origin is used here for checking, never to guide the associators. Select the nearest candidate in origin time and inspect its distance as well. The proposed match below requires an origin difference no larger than 3 seconds and an epicentral separation no larger than 15 km.

In [ ]:
# Choose the closest origin, then also check its distance from the AFAD foreshock.
nearest = events.iloc[np.argmin(np.abs(events.time-foreshock.time))]
separation_m, _, _ = gps2dist_azimuth(
    nearest.latitude, nearest.longitude, foreshock.latitude, foreshock.longitude)
origin_difference = abs(nearest.time-foreshock.time)
foreshock_recovered = origin_difference <= 3 and separation_m <= 15000
print("Foreshock candidate passes comparison:", foreshock_recovered)
print(f"Origin difference: {origin_difference:.2f} s; separation: {separation_m/1000:.1f} km")
event_id = int(nearest.idx)
event_id = 23
print("event_id to check:", event_id)
event = events.set_index("idx").loc[event_id]
event_picks = assignments[assignments.event_idx == event_id]
origin_time = UTCDateTime(float(event.time))

### 3.12 Check a candidate against actual waveforms

Each row is a different station's vertical recording, scaled separately so that weak and strong traces can be seen together. Blue lines mark assigned P arrivals; orange lines mark S. The curves should begin changing near the arrival marks. S may be clearer on horizontal components than on the vertical plot shown here. Travel times differ between stations.

In [ ]:
stations_plot = stations.copy()

stations_plot["distance_km"] = stations_plot.apply(
    lambda row: gps2dist_azimuth(
        event.latitude,
        event.longitude,
        row.latitude,
        row.longitude
    )[0] / 1000.0,
    axis=1
)

# Keep only stations that contributed picks to this event,
# then sort from nearest to farthest.
shown = (
    stations_plot[
        stations_plot.id.isin(event_picks.station)
    ]
    .sort_values("distance_km")
    .head(16)
    .reset_index(drop=True)
)

display(shown[["id", "distance_km"]])

# Optional filter settings.
# Set FILTER = None for unfiltered data.
FILTER = {
    "type": "bandpass",
    "freqmin": 1.0,
    "freqmax": 20.0,
    "corners": 4,
    "zerophase": True,
}

fig, ax = plt.subplots(figsize=(12, 7))

for offset, row in enumerate(shown.itertuples()):

    for trace in raw.select(
        id=f"{row.id}.HHZ"
    ).slice(
        origin_time - 3,
        origin_time + 50
    ):

        tr = trace.copy()

        if FILTER is not None:
            tr.detrend("demean")
            tr.detrend("linear")
            tr.filter(
                FILTER["type"],
                freqmin=FILTER["freqmin"],
                freqmax=FILTER["freqmax"],
                corners=FILTER.get("corners", 4),
                zerophase=FILTER.get("zerophase", True),
            )

        scale = max(np.max(np.abs(tr.data)), 1)

        ax.plot(
            tr.times(reftime=origin_time),
            offset + 0.7 * tr.data / scale,
            lw=0.5,
            color="0.4"
        )

    for arrival in event_picks[
        event_picks.station == row.id
    ].itertuples():

        color = "tab:blue" if arrival.phase == "P" else "tab:orange"

        ax.plot(
            [arrival.time - event.time] * 2,
            [offset - 0.4, offset + 0.4],
            color=color
        )

ax.set(
    yticks=range(len(shown)),
    yticklabels=[
        f"{row.id} ({row.distance_km:.1f} km)"
        for row in shown.itertuples()
    ],
    xlabel="Seconds after candidate origin",
    title="Foreshock candidate: inspect assigned arrivals"
)

ax.invert_yaxis()

plt.show()

## Part 4 — From waveform amplitudes to local magnitude

We have already an estimation of (x,y,z,t) we need a magnitude estimate!

We use the **PyOcto catalog from threshold 0.1** and estimate one local magnitude scale: the western Türkiye relation of **Kılıç et al. (2017)**.

The workflow is:

1. use stations with assigned P and S arrivals,
2. define a quiet noise window and an S-wave signal window, we need a measurement of the signal to noise ratio
3. remove the instrument response and simulate a unit-gain Wood–Anderson instrument,
4. measure a half peak-to-peak amplitude,
5. reject technically unusable or noisy measurements,
6. calculate one station $M_L$ and combine stations with the median.

Kılıç uses the **vertical component when available**, with the larger horizontal amplitude as a fallback. At least three stations are required for a primary event magnitude.

Source: [Kılıç et al. (2017)](https://doi.org/10.1007/s10950-016-9581-9).

In [ ]:
# A unit-gain Wood–Anderson response acting on ground velocity.
WOOD_ANDERSON = {
    "poles": [-5.49779+5.60886j, -5.49779-5.60886j],
    "zeros": [0j],
    "gain": 1.0,
    "sensitivity": 1.0,
}
PRE_FILTER = (0.02, 0.05, 40.0, 45.0)  # Frequency taper corners, Hz.
WAVEFORM_PADDING = 20  # Seconds outside measurement windows to reduce edge effects.

### 4.1 The Kılıç local-magnitude relation

Let $A$ be the unit-gain Wood–Anderson displacement in nanometers and $R$ the hypocentral distance in kilometers:

$$
M_L = \log_{10}(A) + \log_{10}(R) + 0.00190R + c .
$$

We use $c=-1.61$ for vertical measurements and $c=-1.87$ for horizontal measurements.

The equation captures two simple ideas: larger recorded amplitudes imply larger earthquakes, while the same amplitude observed farther from the source also implies a larger earthquake because seismic waves decay with distance.

In [ ]:
def magnitude_value(amplitude_nm, distance_km, vertical=False):
    """Kılıç West 2017 station local magnitude."""
    if amplitude_nm <= 0 or distance_km <= 0:
        raise ValueError("Non-positive amplitude or distance")

    intercept = -1.61 if vertical else -1.87
    return (
        np.log10(amplitude_nm)
        + np.log10(distance_km)
        + 0.00190 * distance_km
        + intercept
    )

### 4.2 Place signal and noise windows using observed arrivals

Only stations with assigned P and S arrivals are used; we do not predict missing picks. Noise spans P-5.5 to P-0.5 s. The signal starts 0.2-1 s before S and ends 5-20 s after S, with both offsets determined by S-P. Another assigned arrival shortens the signal with a 1 s guard. At least 1.6 s total and 0.8 s after S must remain. Known other arrivals near the noise window reject the pair.

In [ ]:
# Put the first assigned P and S for each event/station on the same row.
pairs = assignments.pivot_table(index=["event_idx", "station"], columns="phase",
                                values="time", aggfunc="min").reset_index()
# Require both arrivals for a measured S-wave window.
pairs = pairs.dropna(subset=["P", "S"])
event_lookup = events.set_index("idx")

def measurement_windows(pair):
    p, s = float(pair.P), float(pair.S)
    if s <= p:
        raise ValueError("S must follow P")
    start = s-min(1.0, max(0.2, 0.1*(s-p)))  # Pre-S offset, seconds.
    original_end = s+min(20.0, max(5.0, 2.0*(s-p)))  # Post-S duration.
    noise_start, noise_end = p-5.5, p-0.5
    # Other associated events can contaminate this event's signal or noise.
    other = assignments[(assignments.station == pair.station) &
                        (assignments.event_idx != pair.event_idx)].time
    following = other[(other >= p) & (other <= original_end+1)]
    end = min(original_end, following.min()-1) if len(following) else original_end
    check_windows(other, p, s, start, end, noise_start, noise_end)
    return start, end, noise_start, noise_end, original_end

In [ ]:
def check_windows(other, p, s, start, end, noise_start, noise_end):
    if end-start < 1.6 or end < s+0.8:
        raise ValueError("Overlap leaves incomplete S window")
    if other.between(noise_start-5, noise_end).any():
        raise ValueError("Other arrival near noise window")
    if ((other >= noise_start) & (other <= start) & (other <= p)).any():
        raise ValueError("Other event before target S")

def covering_trace(trace_id, left, right):
    # Join only the short measurement window if it crosses an hourly file boundary.
    segment = raw.select(id=trace_id).slice(UTCDateTime(left), UTCDateTime(right)).copy()
    segment.merge(method=-1)  # Join contiguous matching samples; leave real gaps open.
    if len(segment) != 1:
        raise ValueError("Gap in measurement window")
    trace = segment[0]
    if trace.stats.starttime.timestamp > left+trace.stats.delta or trace.stats.endtime.timestamp < right-trace.stats.delta:
        raise ValueError("Incomplete waveform padding")
    return trace


### 4.3 Instrument correction in two steps

ObsPy first removes the recorded sensor response to obtain ground velocity in m/s. It then simulates a **unit-gain Wood–Anderson** instrument (period 0.8 s, damping 0.7), whose output is displacement in meters. We convert measurement windows to nanometers.

The frequency taper retains 0.05–20 Hz and tapers to zero at 0.02 and 25 Hz. [ObsPy response removal](https://docs.obspy.org/packages/autogen/obspy.core.trace.Trace.remove_response.html).

In [ ]:
def to_wood_anderson(trace, response):
    corrected = trace.copy()  # Never change the raw instrument counts.
    corrected.stats.response = response
    corrected.remove_response(
        output="VEL", pre_filt=PRE_FILTER, water_level=None,
        zero_mean=True, taper=True, taper_fraction=0.05)
    del corrected.stats.response  # applied: no need to copy the full response with every window cut below
    # Simulate a unit-gain Wood–Anderson response, consistent with the Kılıç equation used below.
    corrected.simulate(
        paz_simulate=WOOD_ANDERSON, simulate_sensitivity=False,
        zero_mean=False, taper=False, pitsasim=False, sacsim=True)
    return corrected


def window_nm(trace, start, end):
    cut = trace.slice(UTCDateTime(start), UTCDateTime(end)).copy()
    if len(cut) < 8 or not np.isfinite(cut.data).all() or np.ptp(cut.data) == 0:
        raise ValueError("Invalid or flat window")
    cut.detrend("linear")  # Remove slow drift in this short measurement window.
    return cut.data*1e9, cut  # Meters to nanometers.

### 4.4 Find adjacent peak pairs

Half peak-to-peak means half the difference between neighboring opposite extrema separated by one zero crossing. It is not half the global range of the whole window. This distinction matters when multiple cycles or overlapping earthquakes are present.

In [ ]:
def adjacent_amplitude(values):
    # First locate local maxima and minima by comparing neighboring samples.
    extrema = []
    for index in range(1, len(values)-1):
        peak = values[index] >= values[index-1] and values[index] > values[index+1]
        trough = values[index] <= values[index-1] and values[index] < values[index+1]
        if peak or trough:
            extrema.append(index)
    # Then consider neighboring extrema with exactly one crossing through zero.
    candidates = []
    for left, right in zip(extrema, extrema[1:]):
        signs = np.sign(values[left:right+1])
        signs = signs[signs != 0]  # Zero plateaus must not create extra crossings.
        if np.sum(signs[:-1]*signs[1:] < 0) == 1:
            candidates.append((abs(values[right]-values[left])/2, left, right))
    if not candidates:
        raise ValueError("No adjacent opposite peak pair")
    # Each tuple holds (amplitude, left sample index, right sample index).
    return max(candidates, key=lambda item: item[0])

def signal_quality(raw_trace, values, cut, peak_left, peak_right, start, end):
    # Check the original counts, because correction can hide a saturation plateau.
    raw_values = raw_trace.slice(UTCDateTime(start), UTCDateTime(end)).data
    maximum = np.max(np.abs(raw_values.astype(float)))
    repeated = np.sum(np.abs(raw_values.astype(float)) >= 0.999999*maximum)
    dtype_limit = max(abs(np.iinfo(raw_values.dtype).min), np.iinfo(raw_values.dtype).max)
    if maximum >= 0.98*dtype_limit or repeated >= 3:
        raise ValueError("Possible raw clipping")
    # A peak at the window edge may be truncated, so reject that amplitude.
    rate = cut.stats.sampling_rate
    margin = max(0.2, 2/rate)  # Reject extrema within 0.2 s or two samples of the edge.
    pair_edge = min(peak_left/rate, (len(values)-1-peak_right)/rate)
    maximum_epoch = cut.stats.starttime.timestamp+np.argmax(abs(values))/rate
    if pair_edge <= margin or min(maximum_epoch-start, end-maximum_epoch) <= margin:
        raise ValueError("Amplitude at window edge")

def component_measurement(pair, component, windows):
    start, end, noise_start, noise_end, original_end = windows
    trace_id = f"{pair.station}.HH{component}"
    trace = covering_trace(trace_id, noise_start-WAVEFORM_PADDING, original_end+WAVEFORM_PADDING)
    selection = inventory.select(network=trace.stats.network, station=trace.stats.station,
                                 location=trace.stats.location, channel=trace.stats.channel,
                                 starttime=trace.stats.starttime, endtime=trace.stats.endtime)
    channels = [ch for net in selection for station in net for ch in station]
    channel = channels[0]
    # Correct this component, then measure signal and noise in the same units.
    wa = to_wood_anderson(trace, channel.response)
    signal, cut = window_nm(wa, start, end)
    noise, _ = window_nm(wa, noise_start, noise_end)
    amplitude, left_peak, right_peak = adjacent_amplitude(signal)
    signal_quality(trace, signal, cut, left_peak, right_peak, start, end)
    return measurement_record(pair, component, channel, amplitude, signal, noise)

def measurement_record(pair, component, channel, amplitude, signal, noise):
    event = event_lookup.loc[pair.event_idx]
    # Combine epicentral distance with the source depth and station elevation.
    horizontal = gps2dist_azimuth(event.latitude, event.longitude,
                                  channel.latitude, channel.longitude)[0]/1000
    elevation = channel.elevation-channel.depth  # Receiver height above sea level, meters.
    distance = np.hypot(horizontal, event.depth+elevation/1000)
    # RMS describes the typical size of fluctuations in the quiet window.
    noise_rms = np.sqrt(np.mean(noise**2))
    if noise_rms <= 0 or not np.isfinite(noise_rms):
        raise ValueError("Invalid noise RMS")
    return dict(event_idx=pair.event_idx, station=pair.station, component=component,
                amplitude_nm=amplitude,
                noise_rms_nm=noise_rms, snr=amplitude/noise_rms, distance_km=distance,
                azimuth=channel.azimuth, elevation=elevation,
                latitude=channel.latitude, longitude=channel.longitude,
                sample_rate=channel.sample_rate)


### 4.5 Measure amplitudes and reject unusable windows

We reject gaps, overlapping events, obvious saturation and peaks cut off by window edges. One failed window should not stop measurements at other stations, so the loop records its reason and continues. Missing values are never replaced by zero. SNR is amplitude divided by noise RMS. We apply the SNR ≥ 3 criterion after measuring.

In [ ]:
# Record rejected measurements so their absence remains visible in the final catalog.
component_rows, rejected_rows = [], []
for pair in tqdm(pairs.itertuples(), total=len(pairs), desc="event-station pairs"):  # about a minute
    try:
        windows = measurement_windows(pair)
    except ValueError as error:
        rejected_rows.append(dict(event_idx=pair.event_idx, station=pair.station, reason=str(error)))
        continue
    for component in ["Z", "N", "E"]:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error")  # Response warnings reject this component.
                component_rows.append(component_measurement(pair, component, windows))
        except (ValueError, Warning) as error:
            rejected_rows.append(dict(event_idx=pair.event_idx, station=pair.station,
                                      component=component, reason=str(error)))
components = pd.DataFrame(component_rows)
rejected = pd.DataFrame(rejected_rows)
print(len(components), "technically valid component measurements")
display(rejected.reason.value_counts().head(10))

### 4.6 Select one amplitude per station and combine stations

For the Kılıç relation, use the vertical component when it passes quality control. If vertical is unavailable, use the larger of the two horizontal amplitudes, provided both horizontal components are usable.

Each station contributes at most one $M_L$. We then take the **median station magnitude** for each event, which reduces the influence of unusual station amplitudes. A primary magnitude requires at least three stations.

In [ ]:
def passes_snr(row, minimum):
    # None means accepting all technically usable amplitudes in the sensitivity check.
    return minimum is None or row.snr >= minimum


def select_kilic_amplitude(group, minimum):
    """Choose the amplitude used by the Kılıç relation for one event/station."""
    vertical = group[group.component == "Z"]
    horizontal = group[group.component.isin(["N", "E"])]

    # Prefer Z when it is technically valid and passes the requested SNR threshold.
    if len(vertical) == 1 and passes_snr(vertical.iloc[0], minimum):
        return vertical.iloc[0], True

    # Otherwise require both horizontal components and use the larger amplitude.
    if len(horizontal) == 2:
        best = horizontal.loc[horizontal.amplitude_nm.idxmax()]
        if passes_snr(best, minimum):
            return best, False

    return None, None

In [ ]:
def station_magnitude_rows(minimum=3.0):
    rows = []

    for (event_idx, station), group in components.groupby(["event_idx", "station"]):
        chosen, vertical = select_kilic_amplitude(group, minimum)

        if chosen is None:
            continue

        ml = magnitude_value(
            chosen.amplitude_nm,
            chosen.distance_km,
            vertical=vertical,
        )

        rows.append({
            "event_idx": event_idx,
            "station": station,
            "ml": ml,
            "basis": "Z" if vertical else "Hmax",
            "amplitude_nm": chosen.amplitude_nm,
            "snr": chosen.snr,
            "distance_km": chosen.distance_km,
        })

    return pd.DataFrame(rows)


def aggregate_magnitudes(station_values):
    result = station_values.groupby("event_idx").agg(
        ml=("ml", "median"),
        mean=("ml", "mean"),
        n_sites=("station", "nunique"),
        site_mad=("ml", lambda x: 1.4826*np.median(abs(x-np.median(x)))),
    ).reset_index()

    # Retain provisional estimates, but flag the primary estimate only at >=3 stations.
    result["primary_3sites"] = result.ml.where(result.n_sites >= 3)
    return result


station_magnitudes = station_magnitude_rows(3.0)  # Signal amplitude >= 3 x noise RMS.
magnitudes = aggregate_magnitudes(station_magnitudes)

display(station_magnitudes.head(20))
display(magnitudes.head(20))
print(magnitudes)

### 4.7 Effect of noisy amplitudes on $M_L$

How much does the SNR criterion matter?

We compare the default **SNR ≥ 3** result with a second calculation that accepts every technically usable amplitude. The same measured amplitudes are reused, so this isolates the effect of the SNR selection itself.

The plot shows the five largest absolute changes among events supported by at least three stations in both calculations. A positive bar means accepting noisier amplitudes increased $M_L$; a negative bar means it decreased $M_L$.

In [ ]:
# Reuse the already measured amplitudes; only change the SNR acceptance rule.
no_snr_magnitudes = aggregate_magnitudes(station_magnitude_rows(None))

snr_comparison = magnitudes.merge(
    no_snr_magnitudes,
    on="event_idx",
    suffixes=("_snr3", "_all"),
)

snr_comparison = snr_comparison.dropna(
    subset=["primary_3sites_snr3", "primary_3sites_all"]
)

snr_comparison["change"] = (
    snr_comparison.primary_3sites_all
    - snr_comparison.primary_3sites_snr3
)

extreme = snr_comparison.loc[
    snr_comparison.change.abs().nlargest(5).index
]

fig, ax = plt.subplots(figsize=(10, 3.5), constrained_layout=True)
ax.barh(extreme.event_idx.astype(str), extreme.change)
ax.axvline(0, color="black", lw=0.8)
ax.set(
    xlabel="ML without SNR filter - ML with SNR >= 3",
    ylabel="Event ID",
    title="Largest SNR-related magnitude changes",
)
plt.show()

print("Median absolute change:", round(snr_comparison.change.abs().median(), 3))

### 4.8 Follow one actual amplitude measurement

The next example connects the technical helpers to a waveform.

**Top:** raw counts, with the noise window before P shaded gray and the signal window near S shaded green. The extra samples on both sides are padding for instrument correction.

**Middle:** the corrected Wood–Anderson signal in nanometers. Red points mark the selected neighboring peak and trough; the measured amplitude is half their vertical separation.

**Bottom:** the corrected noise window on its own scale; dashed lines show ± noise RMS. SNR is the measured Wood-Anderson amplitude divided by the noise RMS.

If amplitude doubles at fixed distance, log10(A) increases by about 0.30 and so does ML.

In [ ]:
# Select a clear, technically accepted example for the explanatory figure.
example = components.sort_values("snr", ascending=False).iloc[0]
print(len(components))
pair = pairs[(pairs.event_idx == example.event_idx) & (pairs.station == example.station)].iloc[0]
start, end, noise_start, noise_end, original_end = measurement_windows(pair)
trace_id = f"{example.station}.HH{example.component}"
trace = covering_trace(trace_id, noise_start-WAVEFORM_PADDING, original_end+WAVEFORM_PADDING)
response = inventory.get_response(trace_id, trace.stats.starttime)
wa = to_wood_anderson(trace, response)
# Both windows are in the same physical units, needed for a meaningful SNR.
signal, signal_cut = window_nm(wa, start, end)
noise, noise_cut = window_nm(wa, noise_start, noise_end)
amp, peak_left, peak_right = adjacent_amplitude(signal)
noise_rms = np.sqrt(np.mean(noise**2))
print(trace_id, f"amplitude={amp:.1f} nm, noise RMS={noise_rms:.1f} nm, SNR={amp/noise_rms:.1f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), constrained_layout=True)
# Origin-relative seconds let us label P, S and both windows on the raw trace.
origin = UTCDateTime(float(event_lookup.loc[pair.event_idx, "time"]))
axes[0].plot(trace.times(reftime=origin), trace.data, lw=0.5, color="0.3")
axes[0].axvspan(noise_start-origin.timestamp, noise_end-origin.timestamp, color="0.7", label="Noise")
axes[0].axvspan(start-origin.timestamp, end-origin.timestamp, color="green", alpha=0.2, label="Signal")
for arrival, label, color in [(pair.P, "P", "tab:blue"), (pair.S, "S", "tab:orange")]:
    axes[0].axvline(arrival-origin.timestamp, ls="--", color=color, label=label)
axes[0].set(xlabel="Seconds after candidate origin", ylabel="Counts", title=trace_id)
axes[0].legend(ncol=4)
axes[1].plot(signal_cut.times(), signal, lw=0.8)
axes[1].plot(signal_cut.times()[[peak_left, peak_right]], signal[[peak_left, peak_right]], "ro-")
axes[1].axhline(0, color="0.5", lw=0.5)
axes[1].set(xlabel="Seconds after signal-window start", ylabel="WA displacement (nm)",
            title=f"Red pair: half peak-to-peak amplitude = {amp:.1f} nm")
axes[2].plot(noise_cut.times(), noise, lw=0.7, color="0.4")
for value in [-noise_rms, noise_rms]:
    axes[2].axhline(value, color="orange", ls="--")
axes[2].set(xlabel="Seconds after noise-window start", ylabel="WA displacement (nm)",
            title=f"Noise on its own scale: RMS = {noise_rms:.1f} nm")
plt.show()

## Part 5 — Compare with the AFAD reference catalog

The bundled subset comes from the local AFAD reference catalog. Its foreshock origin is 08:00:22 UTC on 24 September. The original rows are preserved alongside the normalized file, and the same AFAD table loaded at the start is used throughout this notebook. Apply the same geographic box and origin interval to both catalogs.

Match within **3 seconds and 15 km**, using a one-to-one assignment that maximizes the number of eligible pairs and then minimizes their origin/distance cost. Magnitude is not used for matching. Unmatched automatic candidates may include earthquakes absent from AFAD, false detections, or duplicate detections. Establishing which are additional earthquakes requires waveform review.

Source: [AFAD](https://deprem.afad.gov.tr/).

In [ ]:
def match_reference(catalog, reference, seconds=3, kilometers=15):
    # Dummy columns let any automatic event remain unmatched.
    count = len(catalog)
    penalty = 10*(count+len(reference)+1)  # Prioritize number of matches over small costs.
    # Real matches get small costs; extra columns allow an event to stay unmatched.
    costs = np.full((count, len(reference)+count), penalty, dtype=float)
    distances = np.full((count, len(reference)), np.nan)
    for i, event in enumerate(catalog.itertuples()):
        for j, ref in enumerate(reference.itertuples()):
            dt = abs(event.time-ref.time)
            if dt <= seconds:
                km = gps2dist_azimuth(event.latitude, event.longitude, ref.latitude, ref.longitude)[0]/1000
                if km <= kilometers:
                    costs[i, j] = dt/seconds+km/kilometers
                    distances[i, j] = km
    # Choose pairs jointly so that two candidates cannot claim the same AFAD event.
    row_indices, column_indices = linear_sum_assignment(costs)
    return match_rows(catalog, reference, costs, distances, row_indices, column_indices, penalty)

In [ ]:
def match_rows(catalog, reference, costs, distances, rows, columns, penalty):
    matches = []
    # Convert the selected table positions into a readable list of event pairs.
    for i, j in zip(rows, columns):
        if j < len(reference) and costs[i, j] < penalty:
            event, ref = catalog.iloc[i], reference.iloc[j]
            matches.append(dict(event_idx=int(event.idx), reference_idx=j,
                                difference_s=event.time-ref.time, distance_km=distances[i, j]))
    return pd.DataFrame(matches, columns=["event_idx", "reference_idx", "difference_s", "distance_km"])

### 5.1 Check the PyOcto experiment against AFAD

Apply the same 3-second / 15-km one-to-one matching rule to each PyOcto run. Compare recovered AFAD events and unmatched automatic candidates together. Both runs use the same reference events and study interval.

In [ ]:
model_match_rows = []
for label, table in threshold_catalogs.items():
    matches = match_reference(table, afad)  # Same tolerances for both runs.
    model_match_rows.append({"run": label, "AFAD recovered": len(matches),
                             "AFAD missed": len(afad)-len(matches),
                             "unmatched candidates": len(table)-len(matches)})
display(pd.DataFrame(model_match_rows).set_index("run"))

In [ ]:
# Match each automatic catalog separately to the same AFAD table.
reference_matches = {"PyOcto": match_reference(events, afad),
                     "GaMMA": match_reference(gamma_events, afad)}
count_rows = []
for name, table in [("PyOcto", events), ("GaMMA", gamma_events)]:
    number_matched = len(reference_matches[name])
    count_rows.append({"method": name, "matched": number_matched,
                       "afad_unmatched": len(afad)-number_matched,
                       "automatic_unmatched": len(table)-number_matched,
                       "net_count_difference": len(table)-len(afad)})
counts = pd.DataFrame(count_rows).set_index("method")
display(counts)

### 5.2 Check every AFAD event

Each row below is one AFAD event, including events with no automatic match. A strict match passes both the 3-second and 15-km limits. For unmatched rows, the table lists the nearest candidate in origin time as **review only**, together with its time and distance offsets. These rows remain outside the matched count.

In [ ]:
# Keep all reference rows, including the smallest earthquakes and the foreshock.
recovery = afad[["EventID", "origin_time_utc", "magnitude", "type"]].copy()
for name, catalog in [("PyOcto", events), ("GaMMA", gamma_events)]:
    matches = reference_matches[name].set_index("reference_idx")
    for ref_index, ref in afad.iterrows():
        # A strict match takes priority; otherwise show the nearest origin for review.
        matched = ref_index in matches.index
        if matched:
            candidate_id = matches.loc[ref_index, "event_idx"]
            candidate = catalog[catalog.idx == candidate_id].iloc[0]
        else:
            candidate = catalog.iloc[np.argmin(np.abs(catalog.time-ref.time))]
        distance = gps2dist_azimuth(ref.latitude, ref.longitude,
                                   candidate.latitude, candidate.longitude)[0]/1000
        recovery.loc[ref_index, f"{name} status"] = "Matched" if matched else "Review only"
        recovery.loc[ref_index, f"{name} offset (s)"] = candidate.time-ref.time
        recovery.loc[ref_index, f"{name} distance (km)"] = distance
# Large offsets make missing detections distinguishable from location disagreements.
display(recovery.round(2))

### 5.3 Recovery, missed reference events and extra candidates

For each method, **matched + AFAD unmatched = all AFAD events** in the selection. Also, **matched + automatic unmatched = all automatic candidates**. Thus 17 matches plus 13 additional candidates would mean 30 candidates, not 30 recovered AFAD events. The cumulative curves show when the catalog counts diverge.

In [ ]:
# Separate recovered reference events from additional unconfirmed candidates.
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
counts[["matched", "afad_unmatched", "automatic_unmatched"]].plot.bar(ax=axes[0], rot=0)
axes[0].set_ylabel("Event count")
for table, label in [(afad, "AFAD"), (events, "PyOcto"), (gamma_events, "GaMMA")]:
    table = table.sort_values("time")
    axes[1].step((table.time-t0.timestamp)/60, np.arange(1, len(table)+1), where="post", label=label)
axes[1].set(xlim=(0, (t1-t0)/60), xlabel="Minutes after selection start", ylabel="Cumulative candidates")
axes[1].legend()
plt.show()

### 5.4 Compare the matched AFAD magnitudes

For PyOcto events that match AFAD, compare our **Kılıç $M_L$** with the reference magnitude.

Circles show AFAD $M_L$ and stars show AFAD $M_w$. The large foreshock is **$M_w$ 4.6** in the bundled reference file, so that point is a cross-scale comparison rather than an $M_L$ residual.

Hollow symbols mark provisional estimates supported by fewer than three stations. The dashed line indicates numerical equality.

In [ ]:
# Keep every matched AFAD magnitude type, including the foreshock's Mw.
matched_afad = reference_matches["PyOcto"].merge(
    afad,
    left_on="reference_idx",
    right_index=True,
    validate="one_to_one",
)

magnitude_comparison = magnitudes.merge(
    matched_afad,
    on="event_idx",
    validate="one_to_one",
)

limits = [
    min(afad.magnitude.min(), magnitudes.ml.min()) - 0.2,
    max(afad.magnitude.max(), magnitudes.ml.max()) + 0.2,
]

fig, ax = plt.subplots(figsize=(6, 5.5), constrained_layout=True)

for kind, group in magnitude_comparison.groupby("type"):
    marker = "*" if kind.upper() == "MW" else "o"
    ax.scatter(
        group.magnitude,
        group.ml,
        marker=marker,
        s=85,
        label=f"AFAD {kind}",
    )

    provisional = group[group.n_sites < 3]
    if len(provisional):
        ax.scatter(
            provisional.magnitude,
            provisional.ml,
            marker=marker,
            s=85,
            facecolors="white",
            edgecolors="black",
            label="<3 stations",
        )

ax.plot(limits, limits, "k--", lw=0.7)
ax.set(
    xlim=limits,
    ylim=limits,
    xlabel="AFAD magnitude (type in legend)",
    ylabel="Our Kılıç ML",
    title="Matched-event magnitude comparison",
)
ax.legend(fontsize=8)
plt.show()

In [ ]:
# A left join retains matched earthquakes even when their ML could not be measured.
all_magnitude_matches = matched_afad.merge(
    magnitudes,
    on="event_idx",
    how="left",
)

display(
    all_magnitude_matches[
        ["EventID", "magnitude", "type", "ml", "n_sites"]
    ]
)

# Direct residuals are meaningful only when both quantities are ML.
comparable = magnitude_comparison[
    magnitude_comparison.type.str.upper() == "ML"
].copy()

comparable["difference"] = comparable.ml - comparable.magnitude

display(
    comparable[["EventID", "magnitude", "ml", "n_sites", "difference"]]
)

if len(comparable):
    print(
        "Median ML difference (ours - AFAD):",
        round(comparable.difference.median(), 2),
    )

## Part 6 — Your earthquake catalog

The map shows every PyOcto candidate. Circle **area** follows $A = 12 \times 4^{M_L-1} \, \mathrm{points}^2$ using the Kılıç station-median magnitude. Each increase of one magnitude unit multiplies area by four and diameter by two.

Color shows origin relative to the beginning of the sequence. Hollow circles mark magnitudes supported by fewer than three stations; gray crosses mark events without a usable magnitude. The size legend uses exactly the same formula as the plotted events.

In [ ]:
final_catalog = events.copy()
final_catalog["origin_time_utc"] = pd.to_datetime(
    final_catalog.time,
    unit="s",
    utc=True,
)

# Attach the single Kılıç magnitude estimate by event ID.
mag_table = magnitudes.set_index("event_idx")

for field in ["ml", "mean", "n_sites", "site_mad", "primary_3sites"]:
    final_catalog[field] = final_catalog.idx.map(mag_table[field])

final_catalog["n_sites"] = final_catalog["n_sites"].fillna(0).astype(int)

display(
    final_catalog[
        ["idx", "origin_time_utc", "latitude", "longitude", "depth", "ml", "n_sites"]
    ].head(20)
)

In [ ]:
catalog_ml = final_catalog.ml
has_magnitude = catalog_ml.notna()
primary = final_catalog.n_sites >= 3
areas = 12*4**(catalog_ml-1)  # Scatter sizes are areas in points^2.
minutes = (final_catalog.time-t0.timestamp)/60

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

points = ax.scatter(
    final_catalog.longitude[has_magnitude],
    final_catalog.latitude[has_magnitude],
    s=areas[has_magnitude],
    c=minutes[has_magnitude],
    cmap="viridis",
    vmin=0,
    vmax=(t1-t0)/60,
    edgecolors="black",
    linewidths=0.5,
    alpha=0.8,
)

provisional = has_magnitude & ~primary
ax.scatter(
    final_catalog.longitude[provisional],
    final_catalog.latitude[provisional],
    s=areas[provisional],
    facecolors="white",
    edgecolors="black",
    label="<3 stations",
)

ax.scatter(
    final_catalog.longitude[~has_magnitude],
    final_catalog.latitude[~has_magnitude],
    marker="x",
    s=30,
    color="grey",
    label="No magnitude",
)

# Reference circles explain the exponential size scale.
for magnitude in [1, 2, 3, 4]:
    ax.scatter(
        [], [],
        s=12*4**(magnitude-1),
        facecolors="none",
        edgecolors="black",
        label=f"ML {magnitude}",
    )

ax.set(
    xlabel="Longitude (°)",
    ylabel="Latitude (°)",
    title="PyOcto earthquake catalog",
)
ax.set_aspect(map_aspect)
ax.legend(loc="upper left", bbox_to_anchor=(1.22, 1), labelspacing=1.8)
fig.colorbar(points, ax=ax, label="Minutes after sequence start")
plt.show()

### 6.1 Catalog completeness
An earthquake catalog does not detect all earthquakes equally well. Small events are progressively missed as their signal amplitudes approach the background-noise level.
One simple way to visualize this is the cumulative frequency–magnitude distribution, $N(M \geq m)$. If two catalogs cover the same time and region, a catalog containing more small events will extend farther toward lower magnitudes.
With this short dataset we do not attempt a formal estimate of magnitude of completeness $M_c$. We use the distribution only as a qualitative comparison.

In [ ]:
def cumulative_fmd(magnitudes, bin_width=0.1):
    """Return magnitude bins and cumulative number N(M >= m)."""
    mags = np.asarray(magnitudes, dtype=float)
    mags = mags[np.isfinite(mags)]

    mmin = np.floor(mags.min() / bin_width) * bin_width
    mmax = np.ceil(mags.max() / bin_width) * bin_width

    bins = np.arange(
        mmin,
        mmax + bin_width,
        bin_width
    )

    cumulative = np.array([
        np.sum(mags >= m)
        for m in bins
    ])

    return bins, cumulative

In [ ]:
m_ours, n_ours = cumulative_fmd(
    final_catalog["ml"],
    bin_width=0.1
)

m_afad, n_afad = cumulative_fmd(
    afad["magnitude"],
    bin_width=0.1
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.step(
    m_ours,
    n_ours,
    where="post",
    label=f"Our catalog (N={len(final_catalog)})",
    linewidth=2
)

ax.step(
    m_afad,
    n_afad,
    where="post",
    label=f"AFAD (N={len(afad)})",
    linewidth=2
)

ax.set_yscale("log")

ax.set_xlabel("Magnitude")
ax.set_ylabel(r"Cumulative number $N(M \geq m)$")

ax.set_title(
    "Cumulative frequency–magnitude distribution"
)

ax.grid(
    alpha=0.25,
    which="both"
)

ax.legend()

plt.show()

## Part 7 — Where do the earthquakes appear in the spectrogram?

In the ambient-noise practical we looked at a spectrogram of this same morning at **KO.SLVT** and saw short, bright vertical stripes. Now we know which of them are earthquakes: our PyOcto catalog tells us when each event arrived at each station.

For one station we:

1. compute the spectrogram of the vertical component over the whole study interval (20 s windows, 75% overlap),
2. estimate the **background level** at each frequency as the median over time,
3. for every catalog event with picks at this station, take the window from 1 s before its first assigned arrival to 10 s after its last one,
4. compare the strongest power in that window with the background, and call a frequency **detected** where the earthquake rises at least `EXCESS_DB` above it.

The detected band of each event is drawn as a white box on the spectrogram.

In [ ]:
SPEC_STATION = "KO.SLVT."   # same station as in the ambient-noise practical
SPEC_WINDOW_S = 20.0        # spectrogram window length, s
SPEC_OVERLAP_S = 15.0       # overlap between windows, s
FREQ_RANGE = (0.2, 50.0)    # frequencies shown, Hz
EXCESS_DB = 10.0            # how far above background counts as "detected", dB


def compute_spectrogram_db(trace, window_s=SPEC_WINDOW_S, overlap_s=SPEC_OVERLAP_S):
    tr = trace.copy()
    tr.detrend("demean")
    tr.detrend("linear")
    fs = tr.stats.sampling_rate
    freqs, times, sxx = spectrogram(
        tr.data.astype(float), fs=fs,
        nperseg=int(window_s * fs), noverlap=int(overlap_s * fs),
        detrend="linear", scaling="density", mode="psd",
    )
    return freqs, times, 10 * np.log10(np.maximum(sxx, np.finfo(float).tiny))


tr_spec = raw.select(id=f"{SPEC_STATION}.HHZ")[0].copy().trim(t0, t1)
f_spec, t_spec, S_spec_db = compute_spectrogram_db(tr_spec)

in_range = (f_spec >= FREQ_RANGE[0]) & (f_spec <= FREQ_RANGE[1])
f_spec, S_spec_db = f_spec[in_range], S_spec_db[in_range]

# Absolute UTC time (s) at the center of every spectrogram column.
column_times = tr_spec.stats.starttime.timestamp + t_spec

# Typical (background) level at each frequency: the median over the whole interval.
background_db = np.median(S_spec_db, axis=1)
print(tr_spec)

In [ ]:
def detected_band(excess_db, threshold=EXCESS_DB, smooth_hz=0.5):
    # Contiguous frequency band around the strongest excess that stays above the threshold.
    n_smooth = max(1, int(round(smooth_hz / (f_spec[1] - f_spec[0]))))
    smooth = np.convolve(excess_db, np.ones(n_smooth) / n_smooth, mode="same")
    above = smooth >= threshold
    if not above.any():
        return None
    peak = int(np.argmax(np.where(above, smooth, -np.inf)))
    low, high = peak, peak
    while low > 0 and above[low - 1]:
        low -= 1
    while high < len(above) - 1 and above[high + 1]:
        high += 1
    return f_spec[low], f_spec[high], f_spec[peak], smooth[peak], smooth


# One row per catalog event that has at least one assigned pick at this station.
station_arrivals = assignments[assignments.station == SPEC_STATION]
ml_lookup = final_catalog.set_index("idx").ml

band_rows, excess_curves = [], {}
for event_idx, arrivals in station_arrivals.groupby("event_idx"):
    window_start = arrivals.time.min() - 1.0
    window_end = arrivals.time.max() + 10.0
    in_window = (column_times >= window_start) & (column_times <= window_end)
    if not in_window.any():
        continue

    # Strongest power in the event window, relative to the background.
    excess = S_spec_db[:, in_window].max(axis=1) - background_db
    band = detected_band(excess)
    row = dict(event_idx=event_idx, ml=ml_lookup.get(event_idx, np.nan),
               window_start=window_start, window_end=window_end,
               phases="+".join(sorted(arrivals.phase.unique())), detected=band is not None)
    if band is not None:
        row.update(fmin_hz=band[0], fmax_hz=band[1], fpeak_hz=band[2], peak_excess_db=band[3])
        excess_curves[event_idx] = band[4]
    band_rows.append(row)

event_bands = pd.DataFrame(band_rows)
print(f"{len(event_bands)} of {len(events)} catalog events have picks at {SPEC_STATION}; "
      f"{event_bands.detected.sum()} rise at least {EXCESS_DB:.0f} dB above the background.")
display(event_bands[event_bands.detected].round(2).head(39))

In [ ]:
def plot_spectrogram_with_events(start=t0, end=t1, title_note="",
                                 annotate=True, show_magnitude=False):
    # Waveform (top) and spectrogram (bottom).
    # annotate=True draws each event's detected band as a white box;
    # show_magnitude=True also writes its Kılıç ML above the box.
    tr = tr_spec.copy().trim(start, end)
    t_wave = (tr.times() + (tr.stats.starttime - t0)) / 60.0      # minutes after t0
    t_cols = (column_times - t0.timestamp) / 60.0
    shown_cols = (t_cols >= (start - t0) / 60) & (t_cols <= (end - t0) / 60)

    fig, axes = plt.subplots(
        2, 2, figsize=(12, 7), sharex="col",
        gridspec_kw=dict(height_ratios=[1, 2], width_ratios=[1, 0.02], hspace=0.08, wspace=0.03),
    )
    ax_wave, ax_spec, cax = axes[0, 0], axes[1, 0], axes[1, 1]
    axes[0, 1].axis("off")

    data = tr.data - np.mean(tr.data)
    ax_wave.plot(t_wave, data, color="black", linewidth=0.4)
    amp = 2 * np.percentile(np.abs(data), 99)   # clip the largest arrivals, keep small ones visible
    ax_wave.set_ylim(-amp, amp)
    ax_wave.set_ylabel("Counts")
    ax_wave.grid(alpha=0.25)
    ax_wave.set_title(f"{SPEC_STATION}HHZ — {t0.date}{title_note}")

    vmin, vmax = np.percentile(S_spec_db, [5, 99.5])
    mesh = ax_spec.pcolormesh(t_cols[shown_cols], f_spec, S_spec_db[:, shown_cols],
                              shading="auto", vmin=vmin, vmax=vmax)

    visible = event_bands[
        (event_bands.window_end >= start.timestamp) & (event_bands.window_start <= end.timestamp)
    ]
    if not annotate:
        visible = visible.iloc[0:0]
    for ev in visible.itertuples():
        x0 = (ev.window_start - t0.timestamp) / 60
        x1 = (ev.window_end - t0.timestamp) / 60
        if ev.detected:
            ax_spec.add_patch(Rectangle((x0, ev.fmin_hz), x1 - x0, ev.fmax_hz - ev.fmin_hz,
                                        fill=False, edgecolor="white", linewidth=1.2))
            ax_wave.axvspan(x0, x1, color="tab:red", alpha=0.25, lw=0)
            if show_magnitude and np.isfinite(ev.ml):
                ax_spec.text((x0 + x1) / 2, 0.97 * FREQ_RANGE[1], f"{ev.ml:.1f}",
                             color="white", fontsize=7, ha="center", va="top", clip_on=True,
                             bbox=dict(facecolor="black", alpha=0.55, lw=0, pad=1.0))
        else:
            ax_wave.axvspan(x0, x1, color="0.5", alpha=0.25, lw=0)

    ax_spec.set_ylim(*FREQ_RANGE)
    ax_spec.set_xlim((start - t0) / 60, (end - t0) / 60)
    ax_spec.set_xlabel(f"Minutes after {t0.strftime('%H:%M')} UTC")
    ax_spec.set_ylabel("Frequency [Hz]")
    cbar = fig.colorbar(mesh, cax=cax)
    cbar.set_label("PSD [dB, relative counts]")
    plt.show()


# First look: the plain waveform and spectrogram, without any catalog information.
plot_spectrogram_with_events(title_note="  (no annotations)", annotate=False)

Before reading on, try to spot the earthquakes yourself in the plain spectrogram. Then run the next cell, which marks every catalog event.
Set `SHOW_MAGNITUDE = True` to write each event's Kılıç $M_L$ above its box.

In [ ]:
SHOW_MAGNITUDE = True

# Full study interval: each box is narrow, so read it together with the zoom below.
plot_spectrogram_with_events(
    title_note="  (red: detected catalog events, grey: picked but below threshold)",
    show_magnitude=SHOW_MAGNITUDE,
)

# Zoom on the 15 minutes after the foreshock, where the boxes are easy to read.
foreshock_time = UTCDateTime(float(foreshock.time))
plot_spectrogram_with_events(foreshock_time - 120, foreshock_time + 780,
                             "  — zoom on the foreshock", show_magnitude=SHOW_MAGNITUDE)

### 7.1 Which frequencies make an earthquake visible?

Each thin line is one catalog event: how far its strongest power rises above the background at every frequency. The color shows our Kılıç $M_L$. The dashed line is the detection threshold `EXCESS_DB`.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
detected = event_bands[event_bands.detected]
ml_values = detected.ml.to_numpy()
norm = plt.Normalize(np.nanmin(ml_values), np.nanmax(ml_values))

for ev in detected.itertuples():
    color = "0.6" if np.isnan(ev.ml) else plt.cm.viridis(norm(ev.ml))
    ax.plot(f_spec, excess_curves[ev.event_idx], color=color, lw=0.8, alpha=0.8)

ax.axhline(EXCESS_DB, color="black", ls="--", lw=1, label=f"{EXCESS_DB:.0f} dB threshold")
ax.set_xscale("log")
ax.set(xlim=FREQ_RANGE, xlabel="Frequency [Hz]", ylabel="Event − background [dB]",
       title=f"{SPEC_STATION}HHZ: earthquake signal above the background, per event")
ax.grid(alpha=0.25, which="both")
ax.legend(loc="upper right")
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="viridis"), ax=ax, label="Kılıç $M_L$")
plt.show()

#### Exercise 3

* In which frequency band do most earthquakes rise above the background at this station? Compare it with the PPSD and the daily noise cycle from the ambient-noise practical.
* How does the detected band change with magnitude? Why do larger events extend to lower frequencies?
* Some picked events stay below the threshold (gray). Look at them in the zoom: are they really absent from the spectrogram, or hidden by the 20 s window?
* Change `SPEC_STATION` to a station farther from the sequence (e.g. `"TU.CNKL."`). What happens to the highest detected frequencies, and why?

## Part 8 — Wadati diagram: do the events come from the same place?

For a station at hypocentral distance $R$, the P and S arrivals are $t_P = t_0 + R/V_P$ and $t_S = t_0 + R/V_S$. Their difference therefore grows linearly with the P arrival:

$$
t_S - t_P = \left(\frac{V_P}{V_S} - 1\right)(t_P - t_0).
$$

On a **Wadati diagram** we plot $t_S - t_P$ against $t_P$ for every station with both picks. The slope gives $V_P/V_S$, and the line reaches $t_S - t_P = 0$ at the origin time $t_0$, without assuming a velocity model.

To show all events together, we measure $t_P$ from each event's **first P arrival** rather than in absolute time. If the events come from the same place, every station sees the same S−P time for every event, so the points of different events fall on top of each other along one line. An event located elsewhere shifts away from the cloud.

In [ ]:
# One row per event and station with both a P and an S pick (first pick of each phase).
wadati = (
    assignments.pivot_table(index=["event_idx", "station"], columns="phase",
                            values="time", aggfunc="min")
    .dropna(subset=["P", "S"])
    .reset_index()
)
wadati["ts_tp"] = wadati.S - wadati.P
wadati["tp_rel"] = wadati.P - wadati.groupby("event_idx").P.transform("min")
wadati["ml"] = wadati.event_idx.map(ml_lookup)

# Fit one straight line per event (at least 4 stations) to estimate Vp/Vs.
fit_rows = []
for event_idx, group in wadati.groupby("event_idx"):
    if len(group) >= 4:
        slope, intercept = np.polyfit(group.P - group.P.min(), group.ts_tp, 1)
        fit_rows.append(dict(event_idx=event_idx, n_stations=len(group),
                             vp_vs=1 + slope, ts_tp_first=intercept))
wadati_fits = pd.DataFrame(fit_rows)

median_vp_vs = wadati_fits.vp_vs.median()
print(f"{wadati.event_idx.nunique()} events, {len(wadati)} station P–S pairs")
print(f"Median Vp/Vs from {len(wadati_fits)} events with >= 4 stations: {median_vp_vs:.2f} "
      f"(velocity model used by the associators: {6.0 / 3.5:.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)

ordered = wadati.sort_values("ml", na_position="first")   # largest events drawn on top
has_ml = ordered.ml.notna()
if (~has_ml).any():
    ax.scatter(ordered.tp_rel[~has_ml], ordered.ts_tp[~has_ml], s=18, color="0.7",
               edgecolors="0.4", linewidths=0.3, label="No magnitude")
points = ax.scatter(ordered.tp_rel[has_ml], ordered.ts_tp[has_ml], c=ordered.ml[has_ml],
                    cmap="viridis", s=22, edgecolors="black", linewidths=0.3)
fig.colorbar(points, ax=ax, label="Kılıç $M_L$")

# Reference line with the median Vp/Vs, starting at the typical S-P time of the first station.
x_line = np.array([0, wadati.tp_rel.max()])
y0 = wadati_fits.ts_tp_first.median()
ax.plot(x_line, y0 + (median_vp_vs - 1) * x_line, "k--", lw=1,
        label=f"Median fit: Vp/Vs = {median_vp_vs:.2f}")

# Label each station at the median position of its points.
for station, group in wadati.groupby("station"):
    ax.annotate(station.strip("."), (group.tp_rel.median(), group.ts_tp.median()),
                xytext=(6, -3), textcoords="offset points", fontsize=7, color="0.25")

ax.set(xlabel="$t_P$ − first P arrival of the event [s]", ylabel="$t_S - t_P$ [s]",
       title="Wadati diagram: all PyOcto events, colored by magnitude")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")
plt.show()

#### Exercise 4

* Do the points of different events overlap? What does that tell you about where the events are located relative to each other?
* Which stations show the largest scatter in $t_S - t_P$? Could that come from picking errors, from location differences, or from both?
* How does the median $V_P/V_S$ compare with the 6.0 / 3.5 km/s model used by PyOcto and GaMMA? What would a different $V_P/V_S$ change in the locations?
* Are the larger (yellow) events more consistent than the smallest ones? Why might that be?